
# <p style="text-align: center;">Predict Drought Decrees</p>

## Import libraries

In [1]:
import pandas as pd
import numpy as np

import glob
import os

import requests

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import zipfile

import xarray as xr

import folium
from folium.plugins import MarkerCluster

import pandas as pd
from geopy.geocoders import Nominatim

import geopandas as gpd
import cartopy.crs as ccrs

import imageio

from sklearn.neighbors import KDTree
from sklearn.neighbors import BallTree

from sklearn.preprocessing import RobustScaler

## Set parameters

In [2]:
decree_filename_base = 'arrete_'
decrees_folder_name = './../../data/raw/decrees'
communes_folder_name = './../../data/raw/opendatasoft'
weather_folder_name = './../../data/raw/weather/era5'
weather_ncfiles_folder_name = './../../data/raw/weather/era5/ncfiles2'
processed_data_folder_name = './../../data/processed'
output_data_folder_name = './../../data/processed/output'
decrees_filename = 'decrees.parquet'
decrees_locations_filename = 'decrees_locations.parquet'
communes_csv_filename = 'correspondance-code-insee-code-postal-202410.csv'
weather_filename = 'weather2.parquet'
weather_shema_filename = 'weather_shema2.csv'
weather_yearly_filename = 'weather2_yearly.parquet'
weather_yearly_shema_filename = 'weather_shema2_yearly.csv'
drought_filename = 'drought.parquet'
drought_shema_filename = 'drought_shema.csv'
drought_yearly_filename = 'drought_yearly.parquet'
drought_yearly_shema_filename = 'drought_yearly_shema.csv'
drought_commune_clean_filename = 'drought_commune_clean.parquet'
drought_commune_clean_shema_filename = 'drought_commune_clean_shema.csv'
drought_weather_filename = 'drought_weather.parquet'
drought_weather_shema_filename = 'drought_weather_shema.csv'
#weather_zip_file = "193fcd51a8958175843ecbbcaba057c8.zip"

## Load data

| Column    | Description                                      |
|-----------|--------------------------------------------------|
| date      | The date of the observation.                     |
| latitude  | The latitude coordinate of the observation point.|
| longitude | The longitude coordinate of the observation point.|
| number    | A unique identifier for the observation.         |
| expver    | Experiment version number.                       |
| u10       | 10-meter U-component of wind (eastward wind).    |
| v10       | 10-meter V-component of wind (northward wind).   |
| t2m       | 2-meter temperature (air temperature at 2 meters above the surface).|
| sp        | Surface pressure.                                |
| tp        | Total precipitation.                             |
| e         | Evaporation.                                     |
| sro       | Surface runoff.                                  |
| tcrw      | Total column water vapor.                        |
| stl1      | Soil temperature level 1.                        |
| stl2      | Soil temperature level 2.                        |
| slt       | Soil type.                                       |
| swvl1     | Volumetric soil water layer 1.                   |
| swvl2     | Volumetric soil water layer 2.                   |
| cvh       | High vegetation cover.                           |
| cvl       | Low vegetation cover.                            |
| tvh       | High vegetation type.                            |
| tvl       | Low vegetation type.                             |


## Import Data

In [3]:
# read the dataframe from parquet
import pandas as pd

df = pd.read_parquet(os.path.join(processed_data_folder_name, drought_weather_filename))




In [4]:
os.path.join(processed_data_folder_name, drought_weather_shema_filename)

'./../../data/processed/drought_weather_shema.csv'

In [5]:
# Load the schema (data types) from the file
schema = pd.read_csv(os.path.join(processed_data_folder_name, drought_weather_shema_filename), index_col=0).squeeze("columns")

In [6]:
# Apply the schema to the loaded dataframe
df = df.astype(schema.to_dict())

In [7]:
df.head()

,insee_1,nom_commune_1,latitude_1,longitude_1,year_1,decision_1,Code Postal_1,Département_1,Région_1,Code Département_1,...,stl2_mean_2,slt_sum_2,slt_mean_2,swvl1_sum_2,swvl1_mean_2,swvl2_sum_2,swvl2_mean_2,latitude_rad_2,longitude_rad_2,distance_km
0,10002,AILLEVILLE,48.255000,4.694493,2000,0,10200,['AUBE'],['CHAMPAGNE-ARDENNE'],10,...,11.360728,36.0,3.0,4.290780,0.357565,4.225537,0.352128,0.842121,0.082903,4.147094
1,10003,AIX-EN-OTHE,48.197575,3.736328,2000,0,10160,['AUBE'],['CHAMPAGNE-ARDENNE'],10,...,11.673390,24.0,2.0,4.221337,0.351778,4.156231,0.346353,0.842121,0.065450,5.916774
2,10004,ALLIBAUDIERES,48.586400,4.122928,2000,0,10700,['AUBE'],['CHAMPAGNE-ARDENNE'],10,...,11.781789,24.0,2.0,4.192894,0.349408,4.119763,0.343314,0.846485,0.069813,13.198245
3,10005,AMANCE,48.291854,4.487049,2000,0,10140,['AUBE'],['CHAMPAGNE-ARDENNE'],10,...,11.527883,24.0,2.0,4.238762,0.353230,4.164776,0.347065,0.842121,0.078540,4.751647
4,10006,ARCIS-SUR-AUBE,48.527827,4.141971,2000,0,10700,['AUBE'],['CHAMPAGNE-ARDENNE'],10,...,11.784230,24.0,2.0,4.209328,0.350777,4.135861,0.344655,0.846485,0.074176,8.537810


In [8]:
df.columns

Index(['insee_1', 'nom_commune_1', 'latitude_1', 'longitude_1', 'year_1',
       'decision_1', 'Code Postal_1', 'Département_1', 'Région_1',
       'Code Département_1', 'Code Région_1', 'latitude_rad_1',
       'longitude_rad_1', 'latitude_2', 'longitude_2', 'year_2', 't2m_z_sum_2',
       't2m_z_mean_2', 'tp_z_sum_2', 'tp_z_mean_2', 'e_z_sum_2', 'e_z_mean_2',
       'pev_z_sum_2', 'pev_z_mean_2', 'stl1_z_sum_2', 'stl1_z_mean_2',
       'stl2_z_sum_2', 'stl2_z_mean_2', 'slt_z_sum_2', 'slt_z_mean_2',
       'swvl1_z_sum_2', 'swvl1_z_mean_2', 'swvl2_z_sum_2', 'swvl2_z_mean_2',
       't2m_sum_2', 't2m_mean_2', 'tp_sum_2', 'tp_mean_2', 'e_sum_2',
       'e_mean_2', 'pev_sum_2', 'pev_mean_2', 'stl1_sum_2', 'stl1_mean_2',
       'stl2_sum_2', 'stl2_mean_2', 'slt_sum_2', 'slt_mean_2', 'swvl1_sum_2',
       'swvl1_mean_2', 'swvl2_sum_2', 'swvl2_mean_2', 'latitude_rad_2',
       'longitude_rad_2', 'distance_km'],
      dtype='object')

In [9]:
df.dtypes

insee_1                object
nom_commune_1          object
latitude_1            float64
longitude_1           float64
year_1                  int64
decision_1              int64
Code Postal_1          object
Département_1          object
Région_1               object
Code Département_1     object
Code Région_1           int64
latitude_rad_1        float64
longitude_rad_1       float64
latitude_2            float64
longitude_2           float64
year_2                  int32
t2m_z_sum_2           float64
t2m_z_mean_2          float64
tp_z_sum_2            float64
tp_z_mean_2           float64
e_z_sum_2             float64
e_z_mean_2            float64
pev_z_sum_2           float64
pev_z_mean_2          float64
stl1_z_sum_2          float64
stl1_z_mean_2         float64
stl2_z_sum_2          float64
stl2_z_mean_2         float64
slt_z_sum_2           float64
slt_z_mean_2          float64
swvl1_z_sum_2         float64
swvl1_z_mean_2        float64
swvl2_z_sum_2         float64
swvl2_z_me

In [10]:
df.select_dtypes(include=['object']).describe()

,insee_1,nom_commune_1,Code Postal_1,Département_1,Région_1,Code Département_1
count,855388,855388,855388,855388,855388,855388
unique,35612,33103,5843,96,22,96
top,47032,SAINTE-COLOMBE,51300,['PAS-DE-CALAIS'],['RHONE-ALPES'],62
freq,27,336,1104,20616,69242,20616


In [11]:
df.select_dtypes(include=['int32', 'int64']).describe()

,year_1,decision_1,Code Région_1,year_2
count,855388.000000,855388.000000,855388.000000,855388.000000
mean,2011.495720,0.038705,49.248922,2011.495720
std,6.921791,0.192892,25.128815,6.921791
min,2000.000000,0.000000,11.000000,2000.000000
25%,2005.000000,0.000000,25.000000,2005.000000
50%,2011.000000,0.000000,43.000000,2011.000000
75%,2017.000000,0.000000,73.000000,2017.000000
max,2023.000000,1.000000,94.000000,2023.000000


In [12]:
df.select_dtypes(include=['float32', 'float64']).describe()

,latitude_1,longitude_1,latitude_rad_1,longitude_rad_1,latitude_2,longitude_2,t2m_z_sum_2,t2m_z_mean_2,tp_z_sum_2,tp_z_mean_2,...,stl2_mean_2,slt_sum_2,slt_mean_2,swvl1_sum_2,swvl1_mean_2,swvl2_sum_2,swvl2_mean_2,latitude_rad_2,longitude_rad_2,distance_km
count,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,...,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000,855388.000000
mean,47.023746,2.705051,0.820719,0.047212,47.023621,2.705099,4.604819,0.383735,-0.298512,-0.024876,...,11.634466,28.544758,2.378730,3.886423,0.323869,3.803169,0.316931,0.820717,0.047213,9.042541
std,2.164248,2.603619,0.037773,0.045442,2.165805,2.603934,4.181401,0.348450,3.154064,0.262839,...,1.705294,7.758866,0.646572,0.573170,0.047764,0.589023,0.049085,0.037800,0.045447,3.553814
min,42.285250,-5.085345,0.738017,-0.088756,42.500000,-5.000000,-9.260896,-0.771741,-9.733873,-0.811156,...,1.713064,0.000000,0.000000,-0.000040,-0.000003,-0.000050,-0.000004,0.741765,-0.087266,0.060873
25%,45.254133,0.668926,0.789834,0.011675,45.250000,0.750000,1.534026,0.127835,-2.598595,-0.216550,...,10.724335,24.000000,2.000000,3.717030,0.309753,3.591872,0.299323,0.789761,0.013090,6.460910
50%,47.434521,2.647408,0.827889,0.046206,47.500000,2.750000,5.053526,0.421127,-0.255212,-0.021268,...,11.580068,24.000000,2.000000,3.947550,0.328962,3.858700,0.321558,0.829031,0.047997,9.151202
75%,48.848293,4.842895,0.852564,0.084524,48.750000,4.750000,7.871564,0.655964,1.922646,0.160221,...,12.612666,36.000000,3.000000,4.168569,0.347381,4.118390,0.343199,0.850848,0.082903,11.837785
max,51.063772,8.825756,0.891231,0.154038,51.000000,8.500000,17.793317,1.482776,13.873955,1.156163,...,19.520823,48.000000,4.000000,5.718729,0.476561,5.756094,0.479674,0.890118,0.148353,35.531838


In [15]:
def plot_distance_boxplot(df):
    fig = px.box(df, y='distance_km', title='Boxplot of Distance (km)', labels={'distance_km': 'Distance (km)'})
    fig.show()

# Example usage for the plot:
# plot_distance_boxplot(df)

In [16]:
df.columns

Index(['insee_1', 'nom_commune_1', 'latitude_1', 'longitude_1', 'year_1',
       'decision_1', 'Code Postal_1', 'Département_1', 'Région_1',
       'Code Département_1', 'Code Région_1', 'latitude_rad_1',
       'longitude_rad_1', 'latitude_2', 'longitude_2', 'year_2', 't2m_z_sum_2',
       't2m_z_mean_2', 'tp_z_sum_2', 'tp_z_mean_2', 'e_z_sum_2', 'e_z_mean_2',
       'pev_z_sum_2', 'pev_z_mean_2', 'stl1_z_sum_2', 'stl1_z_mean_2',
       'stl2_z_sum_2', 'stl2_z_mean_2', 'slt_z_sum_2', 'slt_z_mean_2',
       'swvl1_z_sum_2', 'swvl1_z_mean_2', 'swvl2_z_sum_2', 'swvl2_z_mean_2',
       't2m_sum_2', 't2m_mean_2', 'tp_sum_2', 'tp_mean_2', 'e_sum_2',
       'e_mean_2', 'pev_sum_2', 'pev_mean_2', 'stl1_sum_2', 'stl1_mean_2',
       'stl2_sum_2', 'stl2_mean_2', 'slt_sum_2', 'slt_mean_2', 'swvl1_sum_2',
       'swvl1_mean_2', 'swvl2_sum_2', 'swvl2_mean_2', 'latitude_rad_2',
       'longitude_rad_2', 'distance_km'],
      dtype='object')

In [18]:
z_columns = [col for col in df.columns if '_z_' in col]
z_columns

['t2m_z_sum_2',
 't2m_z_mean_2',
 'tp_z_sum_2',
 'tp_z_mean_2',
 'e_z_sum_2',
 'e_z_mean_2',
 'pev_z_sum_2',
 'pev_z_mean_2',
 'stl1_z_sum_2',
 'stl1_z_mean_2',
 'stl2_z_sum_2',
 'stl2_z_mean_2',
 'slt_z_sum_2',
 'slt_z_mean_2',
 'swvl1_z_sum_2',
 'swvl1_z_mean_2',
 'swvl2_z_sum_2',
 'swvl2_z_mean_2']

In [19]:
df['decision_1'].describe()

count    855388.000000
mean          0.038705
std           0.192892
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: decision_1, dtype: float64

In [20]:
df['decision_1'].value_counts()

decision_1
0    822280
1     33108
Name: count, dtype: int64

In [22]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
df_sorted = df.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
    ('lat_lon_transformer', LatLonTransformer()),  # Transform latitude and longitude (in radians)
    ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)

# Fit a model using the transformed training features
model = LogisticRegression(class_weight='balanced')  # Handle imbalance with class_weight='balanced'
model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
X_test_transformed = pipeline.transform(X_test)
y_pred = model.predict(X_test_transformed)
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Calculate and print evaluation metrics
# Classification Report (Precision, Recall, F1 Score)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.50      0.67    131880
           1       0.13      0.97      0.24     10582

    accuracy                           0.54    142462
   macro avg       0.56      0.73      0.45    142462
weighted avg       0.93      0.54      0.63    142462

Confusion Matrix:
[[66059 65821]
 [  362 10220]]
ROC-AUC Score: 0.83
Average Precision (PR-AUC): 0.24


In [30]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
df_sorted = df.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
    ('lat_lon_transformer', LatLonTransformer()),  # Transform latitude and longitude (in radians)
    ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)

# Fit a model using the transformed training features
model = LogisticRegression(class_weight='balanced')  # Handle imbalance with class_weight='balanced'
model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
X_test_transformed = pipeline.transform(X_test)
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Adjust threshold
threshold = 0.99
y_pred_threshold = (y_prob >= threshold).astype(int)

# Calculate and print evaluation metrics
print("Classification Report (with adjusted threshold):")
print(classification_report(y_test, y_pred_threshold))

# Confusion Matrix with adjusted threshold
print("Confusion Matrix (with adjusted threshold):")
print(confusion_matrix(y_test, y_pred_threshold))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")


Classification Report (with adjusted threshold):
              precision    recall  f1-score   support

           0       0.93      1.00      0.96    131880
           1       0.15      0.00      0.01     10582

    accuracy                           0.92    142462
   macro avg       0.54      0.50      0.48    142462
weighted avg       0.87      0.92      0.89    142462

Confusion Matrix (with adjusted threshold):
[[131645    235]
 [ 10541     41]]
ROC-AUC Score: 0.83
Average Precision (PR-AUC): 0.24


In [32]:
min_year = 2010
max_year = 2020

In [34]:
import pandas as pd
import numpy as np
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
# min_year = 2000
# max_year = 2020
df_filtered = df[(df['year_1'] >= min_year) & (df['year_1'] <= max_year)]
df_sorted = df_filtered.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
    ('lat_lon_transformer', LatLonTransformer()),  # Transform latitude and longitude (in radians)
    ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Define the Optuna objective function to tune 'C' and 'solver'
def objective(trial):
    # Suggest values for 'C' and 'solver'
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga'])

    # Create the Logistic Regression model with the suggested hyperparameters
    model = LogisticRegression(class_weight='balanced', C=C, solver=solver, max_iter=1000)
    model.fit(X_train_transformed, y_train)

    # Evaluate using F1 score for the minority class
    y_pred = model.predict(X_test_transformed)
    return roc_auc_score(y_test, model.predict_proba(X_test_transformed)[:, 1])

# Create an Optuna study and optimize the objective function
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# Best hyperparameters
print("Best hyperparameters:", study.best_params)

# Fit the best model using the best parameters
best_params = study.best_params
model = LogisticRegression(class_weight='balanced', C=best_params['C'], solver=best_params['solver'], max_iter=1000, solver='lbfgs')
model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Adjust threshold
threshold = 0.3
y_pred_threshold = (y_prob >= threshold).astype(int)

# Calculate and print evaluation metrics
print("Classification Report (with best hyperparameters and adjusted threshold):")
print(classification_report(y_test, y_pred_threshold))

# Confusion Matrix with adjusted threshold
print("Confusion Matrix (with best hyperparameters and adjusted threshold):")
print(confusion_matrix(y_test, y_pred_threshold))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")



[I 2024-10-29 23:31:40,545] A new study created in memory with name: no-name-8536c40e-2c12-4b77-bb21-c2e5cd7f966a
[I 2024-10-29 23:31:43,666] Trial 0 finished with value: 0.708670522627848 and parameters: {'C': 436.10015505943915, 'solver': 'saga'}. Best is trial 0 with value: 0.708670522627848.
[I 2024-10-29 23:31:48,167] Trial 1 finished with value: 0.708695031111154 and parameters: {'C': 877.6272894453169, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.708695031111154.
[I 2024-10-29 23:31:52,176] Trial 2 finished with value: 0.7086641581226233 and parameters: {'C': 22.03536850810869, 'solver': 'saga'}. Best is trial 1 with value: 0.708695031111154.
[I 2024-10-29 23:31:54,831] Trial 3 finished with value: 0.7094657651578633 and parameters: {'C': 0.0640408818768168, 'solver': 'saga'}. Best is trial 3 with value: 0.7094657651578633.
[I 2024-10-29 23:31:58,079] Trial 4 finished with value: 0.7108881875446198 and parameters: {'C': 0.014334350319697887, 'solver': 'lbfgs'}. Best is tria

Best hyperparameters: {'C': 0.0032320862557640895, 'solver': 'lbfgs'}


SyntaxError: keyword argument repeated: solver (1258378911.py, line 96)

In [47]:
selected_departements = ['13']

In [48]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
# min_year = 2000
# max_year = 2020
#selected_departements = ['13']
df_filtered = df[(df['year_1'] >= min_year) & (df['year_1'] <= max_year) & (df['Code Département_1'].isin(selected_departements))]
df_sorted = df_filtered.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
    ('lat_lon_transformer', LatLonTransformer()),  # Transform latitude and longitude (in radians)
    ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Train XGBoost model using XGBClassifier
model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc', use_label_encoder=False)
model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Adjust threshold
threshold = 0.3
y_pred_threshold = (y_prob >= threshold).astype(int)

# Calculate and print evaluation metrics
print("Classification Report (with adjusted threshold):")
print(classification_report(y_test, y_pred_threshold))

# Confusion Matrix with adjusted threshold
print("Confusion Matrix (with adjusted threshold):")
print(confusion_matrix(y_test, y_pred_threshold))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")


Classification Report (with adjusted threshold):
              precision    recall  f1-score   support

           0       0.82      0.97      0.89       222
           1       0.00      0.00      0.00        46

    accuracy                           0.81       268
   macro avg       0.41      0.49      0.45       268
weighted avg       0.68      0.81      0.74       268

Confusion Matrix (with adjusted threshold):
[[216   6]
 [ 46   0]]
ROC-AUC Score: 0.24
Average Precision (PR-AUC): 0.12


In [38]:
df.columns

Index(['insee_1', 'nom_commune_1', 'latitude_1', 'longitude_1', 'year_1',
       'decision_1', 'Code Postal_1', 'Département_1', 'Région_1',
       'Code Département_1', 'Code Région_1', 'latitude_rad_1',
       'longitude_rad_1', 'latitude_2', 'longitude_2', 'year_2', 't2m_z_sum_2',
       't2m_z_mean_2', 'tp_z_sum_2', 'tp_z_mean_2', 'e_z_sum_2', 'e_z_mean_2',
       'pev_z_sum_2', 'pev_z_mean_2', 'stl1_z_sum_2', 'stl1_z_mean_2',
       'stl2_z_sum_2', 'stl2_z_mean_2', 'slt_z_sum_2', 'slt_z_mean_2',
       'swvl1_z_sum_2', 'swvl1_z_mean_2', 'swvl2_z_sum_2', 'swvl2_z_mean_2',
       't2m_sum_2', 't2m_mean_2', 'tp_sum_2', 'tp_mean_2', 'e_sum_2',
       'e_mean_2', 'pev_sum_2', 'pev_mean_2', 'stl1_sum_2', 'stl1_mean_2',
       'stl2_sum_2', 'stl2_mean_2', 'slt_sum_2', 'slt_mean_2', 'swvl1_sum_2',
       'swvl1_mean_2', 'swvl2_sum_2', 'swvl2_mean_2', 'latitude_rad_2',
       'longitude_rad_2', 'distance_km'],
      dtype='object')

In [40]:
import pandas as pd

def get_max_decision_per_year(df):
    # Filter only rows where decision_1 == 1
    df_filtered = df[df['decision_1'] == 1]
    
    # Group by year_1, Département_1, Région_1, Code Département_1, Code Région_1 and count the occurrences
    grouped = df_filtered.groupby(['year_1', 'Département_1', 'Région_1', 'Code Département_1']).size().reset_index(name='count_decision_1')
    
    # Find the group with the maximum count for each year
    max_decision_per_year = grouped.loc[grouped.groupby('year_1')['count_decision_1'].idxmax()]
    
    return max_decision_per_year

# Example usage
# df = your_dataframe
result = get_max_decision_per_year(df)

print(result)


     year_1          Département_1                        Région_1  \
1      2000               ['GERS']               ['MIDI-PYRENEES']   
9      2001        ['PUY-DE-DOME']                    ['AUVERGNE']   
19     2002               ['GERS']               ['MIDI-PYRENEES']   
61     2003      ['HAUTE-GARONNE']               ['MIDI-PYRENEES']   
112    2004   ['BOUCHES-DU-RHONE']  ["PROVENCE-ALPES-COTE D'AZUR"]   
131    2005  ['CHARENTE-MARITIME']            ['POITOU-CHARENTES']   
175    2006   ['BOUCHES-DU-RHONE']  ["PROVENCE-ALPES-COTE D'AZUR"]   
208    2007   ['BOUCHES-DU-RHONE']  ["PROVENCE-ALPES-COTE D'AZUR"]   
231    2008               ['TARN']               ['MIDI-PYRENEES']   
263    2009     ['LOT-ET-GARONNE']                   ['AQUITAINE']   
286    2010            ['GIRONDE']                   ['AQUITAINE']   
308    2011           ['DORDOGNE']                   ['AQUITAINE']   
371    2012               ['GERS']               ['MIDI-PYRENEES']   
390    2013         

In [46]:
import plotly.express as px

def plot_decision_counts_plotly(df, departement='13'):
    # Filter the DataFrame for the specific Département_1
    df_filtered = df[df['Code Département_1'] == departement]
    
    # Group by year_1 and count the occurrences of decision_1 == 1
    counts = df_filtered[df_filtered['decision_1'] == 1].groupby('year_1').size().reset_index(name='count_decision_1')
    
    # Create the line plot using Plotly
    fig = px.line(counts, x='year_1', y='count_decision_1', title=f'Number of Decisions == 1 for Département {departement}',
                  labels={'year_1': 'Year', 'count_decision_1': 'Count of Decisions == 1'}, markers=True)
    fig.show()

# Example usage
# df = your_dataframe
plot_decision_counts_plotly(df)


In [49]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# Custom transformer to extract columns containing '_z_'
class ZColumnExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        # Selecting only columns that contain '_z_' in their name
        z_columns = [col for col in X.columns if '_z_' in col]
        return X[z_columns].copy()

# Custom transformer for latitude and longitude transformations in radians
class LatLonTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # No fitting necessary, just returning the object itself
        return self

    def transform(self, X):
        X = X.copy()
        if 'latitude_rad_1' in X.columns and 'longitude_rad_1' in X.columns:
            lat_rad = X['latitude_rad_1']
            lon_rad = X['longitude_rad_1']
            # Apply sine and cosine transformations
            X['lat_sin'] = np.sin(lat_rad)
            X['lat_cos'] = np.cos(lat_rad)
            X['lon_sin'] = np.sin(lon_rad)
            X['lon_cos'] = np.cos(lon_rad)
            # Dropping original latitude and longitude columns
            X = X.drop(columns=['latitude_rad_1', 'longitude_rad_1'])
        return X

# Assuming df is already defined in your environment
# min_year = 2000
# max_year = 2020
# selected_departements = ['13']
df_filtered = df[(df['year_1'] >= min_year) & (df['year_1'] <= max_year) & (df['Code Département_1'].isin(selected_departements))]
df_sorted = df_filtered.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
    ('lat_lon_transformer', LatLonTransformer()),  # Transform latitude and longitude (in radians)
    ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Train Random Forest model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()  # Using default hyperparameters without tuning
model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Adjust threshold
threshold = 0.3
y_pred_threshold = (y_prob >= threshold).astype(int)

# Calculate and print evaluation metrics
print("Classification Report (with adjusted threshold):")
print(classification_report(y_test, y_pred_threshold))

# Confusion Matrix with adjusted threshold
print("Confusion Matrix (with adjusted threshold):")
print(confusion_matrix(y_test, y_pred_threshold))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")


Classification Report (with adjusted threshold):
              precision    recall  f1-score   support

           0       0.83      1.00      0.90       222
           1       0.00      0.00      0.00        46

    accuracy                           0.82       268
   macro avg       0.41      0.50      0.45       268
weighted avg       0.69      0.82      0.75       268

Confusion Matrix (with adjusted threshold):
[[221   1]
 [ 46   0]]
ROC-AUC Score: 0.18
Average Precision (PR-AUC): 0.12


In [63]:

# Assuming df is already defined in your environment
# min_year = 2000
# max_year = 2020
# selected_departements = ['13']
df_filtered = df[(df['year_1'] >= min_year) & (df['year_1'] <= max_year) & (df['Code Département_1'].isin(selected_departements))]
df_sorted = df_filtered.sort_values(by='year_1')  # Sorting by year to maintain temporal consistency

# Define X and y after sorting
X_sorted = df_sorted.drop(columns=['decision_1'])
y_sorted = df_sorted['decision_1']

# Define a cut-off year for training and testing
cutoff_year = df_sorted['year_1'].quantile(0.8)  # 80% of the data for training, 20% for testing

# Split data into training and testing based on the cutoff year
X_train = X_sorted.loc[df_sorted['year_1'] <= cutoff_year]
y_train = y_sorted.loc[df_sorted['year_1'] <= cutoff_year]
X_test = X_sorted.loc[df_sorted['year_1'] > cutoff_year]
y_test = y_sorted.loc[df_sorted['year_1'] > cutoff_year]

# Creating the pipeline
pipeline = Pipeline([
        ('z_column_extractor', ZColumnExtractor()),    # Extract columns with '_z_'
    ('scaler', StandardScaler())                   # Scaling the selected columns
])

# Fit the pipeline to X_train (features) and transform X_train
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

# Train Random Forest model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(class_weight='balanced')

model.fit(X_train_transformed, y_train)

# Transform the test set and make predictions
y_prob = model.predict_proba(X_test_transformed)[:, 1]  # Probabilities for the positive class

# Adjust threshold
threshold = 0.5
y_pred_threshold = (y_prob >= threshold).astype(int)

# Calculate and print evaluation metrics
print("Classification Report (with adjusted threshold):")
print(classification_report(y_test, y_pred_threshold))

# Confusion Matrix with adjusted threshold
print("Confusion Matrix (with adjusted threshold):")
print(confusion_matrix(y_test, y_pred_threshold))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.2f}")

# Precision-Recall Curve and Average Precision Score (PR-AUC)
average_precision = average_precision_score(y_test, y_prob)
print(f"Average Precision (PR-AUC): {average_precision:.2f}")



Classification Report (with adjusted threshold):
              precision    recall  f1-score   support

           0       0.83      1.00      0.91       222
           1       0.00      0.00      0.00        46

    accuracy                           0.83       268
   macro avg       0.41      0.50      0.45       268
weighted avg       0.69      0.83      0.75       268

Confusion Matrix (with adjusted threshold):
[[222   0]
 [ 46   0]]
ROC-AUC Score: 0.24
Average Precision (PR-AUC): 0.13


/opt/mamba/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/mamba/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/opt/mamba/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



In [64]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best threshold based on F1 score: {best_threshold:.2f}")
y_pred_best_threshold = (y_prob >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_best_threshold))


Best threshold based on F1 score: 0.28
              precision    recall  f1-score   support

           0       0.82      0.97      0.89       222
           1       0.00      0.00      0.00        46

    accuracy                           0.81       268
   macro avg       0.41      0.49      0.45       268
weighted avg       0.68      0.81      0.74       268



/tmp/ipykernel_1761254/861864750.py:4: RuntimeWarning:

invalid value encountered in divide



In [55]:
X_train.describe()

,latitude_1,longitude_1,year_1,Code Région_1,latitude_rad_1,longitude_rad_1,latitude_2,longitude_2,year_2,t2m_z_sum_2,...,stl2_mean_2,slt_sum_2,slt_mean_2,swvl1_sum_2,swvl1_mean_2,swvl2_sum_2,swvl2_mean_2,latitude_rad_2,longitude_rad_2,distance_km
count,1210.000000,1210.000000,1210.000000,1210.0,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,...,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000,1210.000000
mean,43.530041,5.249001,2014.006612,93.0,0.759743,0.091612,43.526446,5.252066,2014.006612,4.954280,...,15.914098,24.476032,2.039670,2.446638,0.203887,2.401680,0.200140,0.759680,0.091666,9.495073
std,0.186999,0.296131,2.581340,0.0,0.003264,0.005168,0.198534,0.301286,2.581340,4.570345,...,0.969679,14.972316,1.247693,0.841019,0.070085,0.805425,0.067119,0.003465,0.005258,3.570726
min,43.192641,4.463964,2010.000000,93.0,0.753854,0.077911,43.250000,4.500000,2010.000000,-5.792103,...,12.376597,0.000000,0.000000,0.732713,0.061059,0.698967,0.058247,0.754855,0.078540,0.996233
25%,43.376423,5.014947,2012.000000,93.0,0.757061,0.087527,43.500000,5.000000,2012.000000,4.020971,...,15.358673,0.000000,0.000000,1.394913,0.116243,1.378271,0.114856,0.759218,0.087266,6.836376
50%,43.517912,5.301576,2014.000000,93.0,0.759531,0.092530,43.500000,5.250000,2014.000000,6.218463,...,15.997050,36.000000,3.000000,2.820539,0.235045,2.677852,0.223154,0.759218,0.091630,9.806249
75%,43.692370,5.508193,2016.000000,93.0,0.762576,0.096136,43.750000,5.500000,2016.000000,7.926165,...,16.621490,36.000000,3.000000,3.112874,0.259406,3.003284,0.250274,0.763582,0.095993,12.226033
max,43.900083,5.753726,2018.000000,93.0,0.766201,0.100421,44.000000,5.750000,2018.000000,12.003717,...,17.804718,36.000000,3.000000,3.521121,0.293427,3.433605,0.286134,0.767945,0.100356,15.891171


In [58]:
y_train.value_counts()

decision_1
0    1060
1     150
Name: count, dtype: int64